<a href="https://colab.research.google.com/github/Ayaz2836/crop-disease-prediction/blob/main/Fertilizer_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Fertilizer Prediction

In [ ]:
import kagglehub
path = kagglehub.dataset_download("gdabhishek/fertilizer-prediction")
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
path_df=os.path.join(path,'Fertilizer Prediction.csv')
df=pd.read_csv(path_df)
df.head()

Using Colab cache for faster access to the 'fertilizer-prediction' dataset.


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea


In [ ]:
df.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,1.0,0.0,37,0,0,Urea
1,29,52,45,1.0,0.0,12,0,36,DAP
2,34,65,62,0.0,0.0,7,9,30,14-35-14
3,32,62,34,1.0,0.0,22,0,20,28-28
4,28,54,46,1.0,0.0,35,0,0,Urea


#Encoding


In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder()
df['Soil Type'] =ohe.fit_transform(df[['Soil Type']]).toarray()
df['Crop Type'] =ohe.fit_transform(df[['Crop Type']]).toarray()

In [ ]:
df.describe()


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
count,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000
mean,30.282828,59.151515,43.181818,0.808081,0.191919,18.909091,3.383838,18.606061
std,3.502304,5.840331,11.271568,0.395814,0.395814,11.599693,5.814667,13.476978
min,25.000000,50.000000,25.000000,0.000000,0.000000,4.000000,0.000000,0.000000
25%,28.000000,54.000000,34.000000,1.000000,0.000000,10.000000,0.000000,9.000000
50%,30.000000,60.000000,41.000000,1.000000,0.000000,13.000000,0.000000,19.000000
75%,33.000000,64.000000,50.500000,1.000000,0.000000,24.000000,7.500000,30.000000
max,38.000000,72.000000,65.000000,1.000000,1.000000,42.000000,19.000000,42.000000


In [ ]:
x=df.drop(columns=['Fertilizer Name'])
y=df['Fertilizer Name']
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=2)

#RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
rf.fit(x_train,y_train)
y_pred=rf.predict(x_test)

In [ ]:
RFC=pd.DataFrame({'Actual':y_test,'Predicted':y_pred})

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix

accuracy_score(y_test,y_pred)

0.9

In [ ]:
confusion_matrix(y_test,y_pred)

array([[1, 1, 1, 0, 0, 0, 0],
       [0, 2, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 4, 0, 0, 0],
       [0, 0, 0, 0, 3, 0, 0],
       [0, 0, 0, 0, 0, 2, 0],
       [0, 0, 0, 0, 0, 0, 5]])

#XGboost

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encode target variable y_train and y_test
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier()
xgb.fit(x_train, y_train_encoded)
y_pred = xgb.predict(x_test)
# If you need the original labels for evaluation or display, decode them back
y_pred_decoded = le.inverse_transform(y_pred)

In [ ]:
accuracy_score(y_test,y_pred_decoded)

0.95

In [ ]:
pd.DataFrame({'Actual':y_test,'Predicted':y_pred_decoded})

,Actual,Predicted
93,14-35-14,14-35-14
30,28-28,28-28
56,Urea,Urea
24,20-20,20-20
16,Urea,Urea
23,Urea,Urea
2,14-35-14,14-35-14
27,Urea,Urea
28,DAP,DAP
13,28-28,28-28


#Crop Disease Prediction

### Inspecting Dataset Contents

Before proceeding with image processing, let's list the contents of the downloaded dataset to understand its directory structure and locate where the actual image files are stored.

In [ ]:
import os
import kagglehub

# Re-download the dataset and get its path
disease_dataset_path = kagglehub.dataset_download("sadmansakibmahi/plant-disease-expert")

# List contents of the downloaded dataset directory
print(os.listdir(disease_dataset_path))

# Correcting the assumption: Images are likely in 'Image Data base' based on os.listdir output
# Further correction: It seems to be a nested 'Image Data base' directory
images_dir = os.path.join(disease_dataset_path, 'Image Data base', 'Image Data base')

if os.path.exists(images_dir):
    print(f"Found images directory: {images_dir}")
    # Display a few image file names to verify
    # Make sure to check if the directory is not empty before listing
    if os.listdir(images_dir):
        sample_images = os.listdir(images_dir)[:5]
        print("Sample image files:", sample_images)
    else:
        print("Images directory is empty.")
else:
    print(f"Images directory not found at: {images_dir}. Please check the dataset structure again.")

Using Colab cache for faster access to the 'plant-disease-expert' dataset.
['Small testing with the dataset', 'plant diseases cure', 'models', 'Plant Diseases Classification Models', 'Image Data base']
Found images directory: /kaggle/input/plant-disease-expert/Image Data base/Image Data base
Sample image files: ['Orange Haunglongbing Citrus greening', 'Waterlogging in plant', 'Potato Late blight', 'potato crop', 'cabbage looper']


### Splitting the Image Dataset into Training and Validation Sets

To prepare our image dataset for training a deep learning model, we will create separate directories for training and validation images. Each of these main directories will contain subdirectories for each disease class, ensuring that the original class structure is preserved. We'll use an 80/20 split, meaning 80% of images for each class will go to the training set and 20% to the validation set.

In [ ]:
import os
import shutil
import random

# Define the base directory for the new split datasets
# Change the base_split_dir to a writable location, e.g., /content/
base_split_dir = os.path.join('/content/', 'split_data')

# Define train and validation directories
train_dir = os.path.join(base_split_dir, 'train')
validation_dir = os.path.join(base_split_dir, 'validation')

# Create base split directories if they don't exist
os.makedirs(train_dir, exist_ok=True)
os.makedirs(validation_dir, exist_ok=True)

print(f"Created base directory for split data: {base_split_dir}")
print(f"Train directory: {train_dir}")
print(f"Validation directory: {validation_dir}")

# Get a list of all disease class names (subdirectories in images_dir)
class_names = [d for d in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, d))]

print(f"Found {len(class_names)} disease classes: {class_names}")

# Define split ratio
train_split_ratio = 0.8

# Iterate through each class to split images
for class_name in class_names:
    class_path = os.path.join(images_dir, class_name)
    images_in_class = [os.path.join(class_path, img) for img in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, img))]
    random.shuffle(images_in_class)

    # Calculate split point
    split_point = int(len(images_in_class) * train_split_ratio)

    train_images = images_in_class[:split_point]
    validation_images = images_in_class[split_point:]

    # Create class-specific directories in train and validation sets
    train_class_dir = os.path.join(train_dir, class_name)
    validation_class_dir = os.path.join(validation_dir, class_name)

    os.makedirs(train_class_dir, exist_ok=True)
    os.makedirs(validation_class_dir, exist_ok=True)

    # Copy images to respective directories
    for img_path in train_images:
        shutil.copy(img_path, train_class_dir)
    for img_path in validation_images:
        shutil.copy(img_path, validation_class_dir)

    print(f"Split {class_name}: {len(train_images)} train images, {len(validation_images)} validation images")

print("Dataset splitting complete!")

Created base directory for split data: /content/split_data
Train directory: /content/split_data/train
Validation directory: /content/split_data/validation
Found 58 disease classes: ['Orange Haunglongbing Citrus greening', 'Waterlogging in plant', 'Potato Late blight', 'potato crop', 'cabbage looper', 'ginger', 'Tomato Tomato mosaic virus', 'Apple Apple scab', 'Peach healthy', 'Tomato healthy', 'potato hollow heart', 'Raspberry healthy', 'Cherry (including_sour) healthy', 'tomato canker', 'Grape healthy', 'bird eye spot in tea', 'Grape Esca Black Measles', 'Common Rust in corn Leaf', 'Tomato Early blight', 'brown blight in tea', 'Apple healthy', 'Apple Black rot', 'Corn (maize) healthy', 'Strawberry healthy', 'Cherry (including sour) Powdery mildew', 'Apple Cedar apple rust', 'healthy tea leaf', 'Tomato Spider mites Two spotted spider mite', 'Potato Early blight', 'Bacterial leaf blight in rice leaf', 'Tomato Septoria leaf spot', 'red leaf spot in tea', 'corn crop', 'Soybean healthy', '

### Verify the split dataset structure

Let's quickly check the number of images in the newly created training and validation directories to ensure the split was successful.

In [ ]:
import os

def count_images_in_directory(directory):
    count = 0
    for root, dirs, files in os.walk(directory):
        count += len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])
    return count

print(f"Total images in training set: {count_images_in_directory(train_dir)}")
print(f"Total images in validation set: {count_images_in_directory(validation_dir)}")

# Optionally, list a few files from a sample class in train/validation
if class_names:
    sample_class = class_names[0]
    sample_train_class_path = os.path.join(train_dir, sample_class)
    sample_val_class_path = os.path.join(validation_dir, sample_class)

    print(f"\nSample files from train/{sample_class}: {os.listdir(sample_train_class_path)[:5]}")
    print(f"Sample files from validation/{sample_class}: {os.listdir(sample_val_class_path)[:5]}")

Total images in training set: 159712
Total images in validation set: 39953

Sample files from train/Orange Haunglongbing Citrus greening: ['Orange_Haunglongbing_Citrus_greening15943.jpg', 'Orange_Haunglongbing_Citrus_greening49060.jpg', 'Orange_Haunglongbing_Citrus_greening20732.jpg', 'Orange_Haunglongbing_Citrus_greening32344.jpg', 'Orange_Haunglongbing_Citrus_greening29315.jpg']
Sample files from validation/Orange Haunglongbing Citrus greening: ['Orange_Haunglongbing_Citrus_greening30797.jpg', 'Orange_Haunglongbing_Citrus_greening10565.jpg', 'Orange_Haunglongbing_Citrus_greening16122.jpg', 'Orange_Haunglongbing_Citrus_greening43519.jpg', 'Orange_Haunglongbing_Citrus_greening13467.jpg']


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define image parameters
img_width, img_height = 128, 128 # You can adjust this based on your image sizes
batch_size = 15
epochs = 5# Start with a few epochs, can be increased later

In [ ]:
# Image data generators for training and validation
train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

validation_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical' # Use 'categorical' for multi-class classification
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

Found 159712 images belonging to 58 classes.
Found 39953 images belonging to 58 classes.


In [ ]:
# Build the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(len(train_generator.class_indices), activation='softmax') # Output layer with number of classes
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 58)             │        29,754 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,968,570 (49.47 MB)

 Trainable params: 12,968,570 (49.47 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

Epoch 1/5
 7875/31942 ━━━━━━━━━━━━━━━━━━━━ 3:03:19 457ms/step - accuracy: 0.4685 - loss: 2.0772

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()